# Testing and Code Quality

Notebook for **Chapter 10** of the *python* study document. Run the cells top to bottom. Read the full explanation on the site: https://dadipavan.github.io/Learning/study/python/ch10.html

Tip: change a number, predict the output, then run the cell again.

## 10.1 Why We Test, and the Testing Pyramid

In [ ]:
import unittest
def add(a, b):
    return a + b
class TestAdd(unittest.TestCase):
    def test_ints(self):
        self.assertEqual(add(2, 3), 5)
    def test_floats(self):
        self.assertAlmostEqual(add(0.1, 0.2), 0.3)
    def test_bad(self):
        self.assertEqual(add(2, 2), 5)
    def test_type(self):
        with self.assertRaises(TypeError):
            add("a", 1)
if __name__ == "__main__":
    unittest.main(verbosity=2)

In [ ]:
# slug.py
import re
def slugify(text):
    text = text.strip().lower()
    text = re.sub(r"[^a-z0-9]+", "-", text)
    return text.strip("-")

In [ ]:
# test_slug.py
import unittest
from slug import slugify
class T(unittest.TestCase):
    def test_basic(self):
        self.assertEqual(slugify("Hello World"), "hello-world")
    def test_punct(self):
        self.assertEqual(slugify("  Python, 101!  "), "python-101")
    def test_empty(self):
        self.assertEqual(slugify(""), "")
    def test_subtests(self):
        cases = [("A  B", "a-b"), ("--x--", "x"), ("C++ Tips", "c-tips")]
        for raw, want in cases:
            with self.subTest(raw=raw):
                self.assertEqual(slugify(raw), want)

## 10.2 pytest: Plain Asserts, Fixtures and Parametrize

In [ ]:
# test_slug_pytest.py   (not run here)
import pytest
from slug import slugify

def test_basic():
    assert slugify("Hello World") == "hello-world"

def test_empty():
    assert slugify("") == ""

def test_none_raises():
    with pytest.raises(AttributeError):
        slugify(None)

def test_float_close():
    assert 0.1 + 0.2 == pytest.approx(0.3)

In [ ]:
# not run here
import pytest
from slug import slugify

@pytest.mark.parametrize(
    "raw, want",
    [
        ("A  B", "a-b"),
        ("--x--", "x"),
        ("C++ Tips", "c-tips"),
        ("", ""),
    ],
)
def test_slugify_cases(raw, want):
    assert slugify(raw) == want

In [ ]:
# not run here
import sqlite3
import pytest

@pytest.fixture
def conn():
    c = sqlite3.connect(":memory:")
    c.execute("CREATE TABLE users(id INTEGER PRIMARY KEY, name TEXT UNIQUE)")
    yield c            # the test runs here
    c.close()          # clean-up, even if the test failed

def test_insert(conn):
    conn.execute("INSERT INTO users(name) VALUES (?)", ("ravi",))
    rows = conn.execute("SELECT name FROM users").fetchall()
    assert rows == [("ravi",)]

In [ ]:
import tempfile, pathlib, unittest, json
def save(path, obj): path.write_text(json.dumps(obj))
def load(path): return json.loads(path.read_text())
class T(unittest.TestCase):
    def test_roundtrip(self):
        with tempfile.TemporaryDirectory() as d:
            p = pathlib.Path(d) / "x.json"
            save(p, {"a": [1, 2]})
            self.assertEqual(load(p), {"a": [1, 2]})
unittest.main(argv=["x"], verbosity=1)

----------------------------------------------------------------------
Ran 1 test in 0.010s

OK


## 10.3 Mocks, Fakes, monkeypatch and Database Tests

In [ ]:
import unittest
from unittest import mock

def fetch_price(client, sku):
    resp = client.get(f"/price/{sku}")
    if resp["status"] != 200:
        raise ValueError("bad status")
    return resp["body"]["price"]

def total(client, skus):
    return sum(fetch_price(client, s) for s in skus)

class TestMock(unittest.TestCase):
    def test_total_with_mock(self):
        client = mock.Mock()
        client.get.side_effect = [
            {"status": 200, "body": {"price": 100}},
            {"status": 200, "body": {"price": 250}},
        ]
        self.assertEqual(total(client, ["a", "b"]), 350)
        self.assertEqual(client.get.call_count, 2)
        client.get.assert_any_call("/price/a")
    def test_error(self):
        client = mock.Mock()
        client.get.return_value = {"status": 500, "body": {}}
        with self.assertRaises(ValueError):
            fetch_price(client, "x")

class FakeClient:
    def __init__(self, prices): self.prices = prices
    def get(self, path):
        sku = path.rsplit("/", 1)[1]
        if sku in self.prices:
            return {"status": 200, "body": {"price": self.prices[sku]}}
        return {"status": 404, "body": {}}

class TestFake(unittest.TestCase):
    def test_total(self):
        self.assertEqual(total(FakeClient({"a": 1, "b": 2}), ["a", "b"]), 3)
    def test_missing(self):
        with self.assertRaises(ValueError):
            total(FakeClient({}), ["zzz"])

In [ ]:
# not run here: pytest style
def test_user_from_env(monkeypatch):
    monkeypatch.setenv("APP_USER", "asha")
    assert home_name() == "asha"

def test_with_mocker(mocker):            # needs pytest-mock
    mocker.patch("time.time", return_value=1000.0)

In [ ]:
from unittest import mock
import bill, util
with mock.patch("util.rate", return_value=100.0):
    print("patch util.rate  ->", bill.to_inr(2))
with mock.patch("bill.rate", return_value=100.0):
    print("patch bill.rate  ->", bill.to_inr(2))
print("no patch         ->", bill.to_inr(2))

patch util.rate  -> 166.0
patch bill.rate  -> 200.0
no patch         -> 166.0


In [ ]:
# This cell is meant to fail: read the error message, that is the lesson.
from unittest import mock
m2 = mock.Mock(spec=["get"])
try:
    m2.post()
except AttributeError as e:
    print("AttributeError:", e)
m3 = mock.Mock()
m3.assert_called_once_wiht()

AttributeError: Mock object has no attribute 'post'
...
AttributeError: 'assert_called_once_wiht' is not a valid assertion. Use a spec for the mock if 'assert_called_once_wiht' is meant to be an attribute.. Did you mean: 'assert_called_once_with'?


In [ ]:
import sqlite3, unittest
class Repo:
    def __init__(self, conn):
        self.conn = conn
        conn.execute("CREATE TABLE users(id INTEGER PRIMARY KEY, name TEXT UNIQUE)")
    def add(self, name):
        cur = self.conn.execute("INSERT INTO users(name) VALUES (?)", (name,))
        return cur.lastrowid
    def names(self):
        return [r[0] for r in self.conn.execute("SELECT name FROM users ORDER BY id")]
class TestRepo(unittest.TestCase):
    def setUp(self):
        self.conn = sqlite3.connect(":memory:")
        self.repo = Repo(self.conn)
    def tearDown(self):
        self.conn.close()
    def test_add(self):
        self.repo.add("ravi"); self.repo.add("meena")
        self.assertEqual(self.repo.names(), ["ravi", "meena"])
    def test_unique(self):
        self.repo.add("ravi")
        with self.assertRaises(sqlite3.IntegrityError):
            self.repo.add("ravi")

## 10.4 TDD and Coverage

In [ ]:
# test_fizz.py
import unittest
from fizz import fizzbuzz
class T(unittest.TestCase):
    def test_plain(self): self.assertEqual(fizzbuzz(1), "1")
    def test_fizz(self): self.assertEqual(fizzbuzz(3), "Fizz")
    def test_buzz(self): self.assertEqual(fizzbuzz(5), "Buzz")
    def test_both(self): self.assertEqual(fizzbuzz(15), "FizzBuzz")

In [ ]:
def fizzbuzz(n):
    out = ""
    if n % 3 == 0:
        out += "Fizz"
    if n % 5 == 0:
        out += "Buzz"
    return out or str(n)

....
----------------------------------------------------------------------
Ran 4 tests in 0.000s

OK


## 10.5 Code Quality: Formatters, Linters, Type Checkers and pre-commit

In [ ]:
def add_tag(tag, tags=[]):
    tags.append(tag)
    return tags
print(add_tag("a"), add_tag("b"))

def add_tag_ok(tag, tags=None):
    if tags is None:
        tags = []
    tags.append(tag)
    return tags
print(add_tag_ok("a"), add_tag_ok("b"))

['a', 'b'] ['a', 'b']
['a'] ['b']


In [ ]:
def double(x: int) -> int:
    return x * 2

double("5")

## 10.6 Property-Based Tests, Fake Clocks, Mocks and Flaky Tests

Example-based tests check the cases you thought of. Bugs live in the cases you did not. This section shows four techniques that find those bugs: property-based testing, fake clocks, careful use of mocks, and a method for taming flaky tests. Everything runs with the standard library only. The popular library for property-based testing is Hypothesis; it is not installed here, so its API is *not run here*, and we build the core idea by hand in 15 lines so you see exactly how it works.

In [ ]:
import random

def chunk(items, size):
    out = []
    for i in range(0, len(items) - 1, size):   # bug: should be len(items)
        out.append(items[i:i + size])
    return out

def check(items, size):
    pieces = chunk(items, size)
    flat = [x for p in pieces for x in p]
    return flat == items and all(1 <= len(p) <= size for p in pieces)

def shrink(items, size):
    changed = True
    while changed:
        changed = False
        for i in range(len(items)):
            smaller = items[:i] + items[i + 1:]
            if not check(smaller, size):
                items, changed = smaller, True
                break
    return items

rng = random.Random(42)
for trial in range(1, 201):
    size = rng.randint(1, 4)
    items = [rng.randint(0, 9) for _ in range(rng.randint(0, 10))]
    if not check(items, size):
        print("failed on trial", trial, "size", size, "items", items)
        print("shrunk to", shrink(items, size), "size", size)
        break

failed on trial 3 size 1 items [6, 0, 0, 1, 3, 3, 8, 9, 0]
shrunk to [0] size 1


In [ ]:
import time

class FakeClock:
    def __init__(self):
        self.now = 0.0
        self.sleeps = []
    def sleep(self, s):
        self.sleeps.append(s)
        self.now += s

def retry(fn, tries, base, clock):
    for attempt in range(tries):
        try:
            return fn()
        except ConnectionError:
            if attempt == tries - 1:
                raise
            clock.sleep(base * 2 ** attempt)

calls = {"n": 0}
def flaky():
    calls["n"] += 1
    if calls["n"] < 4:
        raise ConnectionError
    return "ok"

clock = FakeClock()
t0 = time.perf_counter()
print(retry(flaky, 5, 1.0, clock), clock.sleeps, clock.now)
print("real time under 0.1 s:", time.perf_counter() - t0 < 0.1)

ok [1.0, 2.0, 4.0] 7.0
real time under 0.1 s: True


In [ ]:
from unittest import mock
import datetime, random

def greeting(now=None):
    now = now or datetime.datetime.now()
    return "good morning" if now.hour < 12 else "good afternoon"

print(greeting(datetime.datetime(2026, 1, 1, 9)))
print(greeting(datetime.datetime(2026, 1, 1, 15)))

def roll():
    return random.randint(1, 6)

with mock.patch("random.randint", return_value=4) as m:
    print(roll(), m.call_args)

def fetch(client, url):
    r = client.get(url)
    return r.status if r.status == 200 else None

client = mock.Mock()
client.get.return_value.status = 500
print(fetch(client, "http://x"), client.get.call_count)
client.get.side_effect = [mock.Mock(status=200)]
print(fetch(client, "http://x"))

good morning
good afternoon
4 call(1, 6)
None 1
200


In [ ]:
import random

rng = random.Random(1)
passed = 0
for _ in range(100):
    xs = [rng.randint(-50, 50) for _ in range(rng.randint(0, 12))]
    if sorted(xs) == sorted(sorted(xs)):
        passed += 1
print(passed)

100


## Chapter recap

## Practice set

Problem 10.P.1 (easy) What is the output of `slugify(" Python, 101! ")` with the function from 10.1?

In [ ]:
import pytest
from fizz import fizzbuzz

@pytest.mark.parametrize("n, want", [(1, "1"), (3, "Fizz"), (5, "Buzz"), (15, "FizzBuzz")])
def test_fizzbuzz(n, want):
    assert fizzbuzz(n) == want

In [ ]:
# not run here
import pytest
shared = []

@pytest.fixture
def items():
    return shared

## Prompt pack

---
Done? Do the **Practice set** at the end of the chapter, then clear your Review cards on the site.

https://dadipavan.github.io/Learning/study/python/ch10.html